# Seoul Grid Vitality Project - Colab 실행 노트북

이 노트북은 본 프로젝트를 Google Colab에서 실행하기 위한 셀 모음입니다.

- 권장 런타임: `Runtime > Change runtime type > GPU`
- 프로젝트 전체 폴더를 zip으로 압축해서 업로드하거나, Google Drive에 올린 뒤 마운트해서 사용합니다.
- 현재 프로젝트는 `covered-area analysis` 범위입니다. 서울 전역 완전 커버리지 결과로 해석하지 않습니다.

## 1. 런타임/패키지 설치

In [ ]:
!python --version
!nvidia-smi || true

# Core ML/data packages + spatial packages used by the project artifact scripts.
!pip -q install pandas numpy scikit-learn scipy torch requests matplotlib geopandas pyproj shapely pyshp


## 2-A. 프로젝트 zip 업로드 방식

로컬의 `DeepLearning` 폴더를 zip으로 압축한 뒤 업로드합니다. 예: `DeepLearning.zip`

In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile, os, shutil
from pathlib import Path

zip_name = next(iter(uploaded.keys()))
PROJECT_ROOT = Path('/content/DeepLearning')

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content')

# zip 내부가 DeepLearning/로 감싸져 있지 않은 경우를 보정합니다.
if not PROJECT_ROOT.exists():
    candidates = [p for p in Path('/content').iterdir() if p.is_dir() and (p / 'seoul_grid_vitality_pipeline.py').exists()]
    if candidates:
        PROJECT_ROOT = candidates[0]

print('PROJECT_ROOT =', PROJECT_ROOT)
%cd {PROJECT_ROOT}

## 2-B. Google Drive 마운트 방식

Drive에 프로젝트 폴더를 올려둔 경우, 위 업로드 셀 대신 아래 셀을 사용하세요.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# from pathlib import Path
# PROJECT_ROOT = Path('/content/drive/MyDrive/DeepLearning')  # 본인 Drive 경로로 수정
# %cd {PROJECT_ROOT}
# print('PROJECT_ROOT =', PROJECT_ROOT)

## 3. 프로젝트 파일 확인

In [ ]:
from pathlib import Path
import pandas as pd

required_files = [
    'seoul_grid_vitality_pipeline.py',
    'fetch_seoul_realtime_api_to_csv.py',
    'build_live_correction_sequence.py',
    'build_citywide_vitality_artifacts.py',
    'build_base_coverage_map.py',
    'data/base_train.csv',
    'data/base_infer.csv',
    'data/correction_train.csv',
    'data/correction_infer_live_spatial_sequence.csv',
    'data/final_actual.csv',
    'data/grid_place_mapping_spatial.csv',
    'data/seoul_live_place_catalog_official.csv',
    'data/seoul_municipalities_geo.json',
]

missing = [f for f in required_files if not Path(f).exists()]
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))

for f in required_files:
    print(f, Path(f).stat().st_size)

print()
print('base_train preview')
display(pd.read_csv('data/base_train.csv').head())


## 4. Colab 실행 환경 변수 설정

기본 출력은 `/content/DeepLearning/outputs_colab`에 저장합니다.

In [ ]:
import json
import os
import pandas as pd
import subprocess
import sys
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

# Set this to False only for offline/reproducible runs with the packaged live sequence CSV.
USE_LIVE_API = True

DATA_DIR = Path('./data')
PIPELINE_OUTPUT_DIR = Path('./outputs_colab')
CITYWIDE_OUTPUT_DIR = Path('./outputs/citywide_vitality')
COVERAGE_OUTPUT_DIR = Path('./outputs/coverage')
PACKAGED_LIVE_SEQUENCE = DATA_DIR / 'correction_infer_live_spatial_sequence.csv'
LIVE_API_RAW = DATA_DIR / 'correction_infer_live_spatial_api.csv'
LIVE_API_SEQUENCE = DATA_DIR / 'correction_infer_live_spatial_sequence_api.csv'
API_USAGE_MANIFEST = Path('api_usage_manifest.json')

os.environ['SEOUL_GRID_DATA_DIR'] = str(DATA_DIR)
os.environ['SEOUL_GRID_OUTPUT_DIR'] = str(PIPELINE_OUTPUT_DIR)
os.environ['SEOUL_GRID_ANALYSIS_SCOPE'] = 'Current covered-area analysis (Colab run)'
os.environ['SEOUL_GRID_SCOPE_MODE'] = 'covered_area'
os.environ['SEOUL_GRID_BASE_SCORES_CSV'] = str(PIPELINE_OUTPUT_DIR / 'base_scores.csv')
os.environ['SEOUL_GRID_LIVE_FINAL_SCORES_CSV'] = str(PIPELINE_OUTPUT_DIR / 'final_scores.csv')
os.environ['SEOUL_GRID_SPATIAL_MAPPING_CSV'] = str(DATA_DIR / 'grid_place_mapping_spatial.csv')
os.environ['SEOUL_GRID_CITYWIDE_OUTPUT_DIR'] = str(CITYWIDE_OUTPUT_DIR)
os.environ['SEOUL_GRID_COVERAGE_OUTPUT_DIR'] = str(COVERAGE_OUTPUT_DIR)

PIPELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CITYWIDE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COVERAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

api_mode = 'live_api' if USE_LIVE_API else 'packaged_sequence'


def run_step(command, label):
    print(f'\n--- {label} ---')
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed. Check the output above for the original traceback and API diagnostics.')

if USE_LIVE_API:
    os.environ['SEOUL_RT_API_URL'] = os.environ.get('SEOUL_RT_API_URL', 'http://openapi.seoul.go.kr:8088')
    os.environ['SEOUL_RT_API_TYPE'] = os.environ.get('SEOUL_RT_API_TYPE', 'json')
    os.environ['SEOUL_RT_MAPPING_PATH'] = str(DATA_DIR / 'grid_place_mapping_spatial.csv')
    os.environ['SEOUL_RT_OUTPUT'] = str(LIVE_API_RAW)

    if not os.environ.get('SEOUL_RT_API_KEY', '').strip():
        os.environ['SEOUL_RT_API_KEY'] = getpass('서울시 실시간 API 인증키를 입력하세요: ').strip()
    if not os.environ['SEOUL_RT_API_KEY']:
        raise ValueError('USE_LIVE_API=True 이므로 SEOUL_RT_API_KEY가 필요합니다.')

    print('Fetching Seoul live API data...')
    run_step([sys.executable, 'fetch_seoul_realtime_api_to_csv.py'], 'Fetch Seoul live API data')

    if not LIVE_API_RAW.exists() or LIVE_API_RAW.stat().st_size == 0:
        raise FileNotFoundError(f'API raw output was not created: {LIVE_API_RAW}')

    live_raw_rows = len(pd.read_csv(LIVE_API_RAW))
    if live_raw_rows == 0:
        raise ValueError('서울시 API 결과가 0건입니다. API 키, 호출 제한, 장소 코드 매핑을 확인하세요.')

    os.environ['SEOUL_GRID_LIVE_CORRECTION_CSV'] = str(LIVE_API_RAW)
    os.environ['SEOUL_GRID_LIVE_SEQUENCE_CSV'] = str(LIVE_API_SEQUENCE)
    print('Building LSTM-ready live correction sequence...')
    run_step([sys.executable, 'build_live_correction_sequence.py'], 'Build LSTM-ready live correction sequence')

    if not LIVE_API_SEQUENCE.exists() or LIVE_API_SEQUENCE.stat().st_size == 0:
        raise FileNotFoundError(f'Live API sequence output was not created: {LIVE_API_SEQUENCE}')

    live_sequence_rows = len(pd.read_csv(LIVE_API_SEQUENCE))
    if live_sequence_rows == 0:
        raise ValueError('API 기반 LSTM sequence가 0건입니다. live raw CSV와 history correction 데이터를 확인하세요.')

    os.environ['SEOUL_GRID_CORRECTION_INFER_CSV'] = str(LIVE_API_SEQUENCE)
else:
    if not PACKAGED_LIVE_SEQUENCE.exists():
        raise FileNotFoundError(f'Packaged live sequence is missing: {PACKAGED_LIVE_SEQUENCE}')
    live_raw_rows = None
    live_sequence_rows = len(pd.read_csv(PACKAGED_LIVE_SEQUENCE))
    if live_sequence_rows == 0:
        raise ValueError(f'Packaged live sequence is empty: {PACKAGED_LIVE_SEQUENCE}')
    os.environ['SEOUL_GRID_CORRECTION_INFER_CSV'] = str(PACKAGED_LIVE_SEQUENCE)

manifest = {
    'api_mode': api_mode,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'api_key_set': bool(os.environ.get('SEOUL_RT_API_KEY', '').strip()),
    'api_url': os.environ.get('SEOUL_RT_API_URL', ''),
    'mapping_csv': os.environ['SEOUL_GRID_SPATIAL_MAPPING_CSV'],
    'live_raw_csv': str(LIVE_API_RAW) if USE_LIVE_API else None,
    'live_raw_rows': live_raw_rows,
    'correction_infer_csv': os.environ['SEOUL_GRID_CORRECTION_INFER_CSV'],
    'live_sequence_rows': live_sequence_rows,
    'pipeline_output_dir': os.environ['SEOUL_GRID_OUTPUT_DIR'],
    'base_scores_csv': os.environ['SEOUL_GRID_BASE_SCORES_CSV'],
    'final_scores_csv': os.environ['SEOUL_GRID_LIVE_FINAL_SCORES_CSV'],
}
API_USAGE_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Pipeline output dir:', os.environ['SEOUL_GRID_OUTPUT_DIR'])
print('Citywide output dir:', os.environ['SEOUL_GRID_CITYWIDE_OUTPUT_DIR'])
print('Coverage output dir:', os.environ['SEOUL_GRID_COVERAGE_OUTPUT_DIR'])


## 5. 기본 학습/추론 파이프라인 실행

Base MLP와 Correction LSTM을 학습하고 `final_scores.csv`, `metrics_summary.json`, heatmap을 생성합니다.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, 'seoul_grid_vitality_pipeline.py'],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(
        'seoul_grid_vitality_pipeline.py failed. 위 출력의 마지막 Python traceback을 확인하세요.'
    )


## 6. 공간 확장/시각화 산출물 생성

현재 프로젝트에 포함된 산출 스크립트로 covered-area heatmap, 요약 CSV, 커버리지 지도 등을 생성합니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Build place-grid mapping only when it is missing. The uploaded project already includes
# data/grid_place_mapping_spatial.csv, so this normally skips the spatial join step.
if not Path('data/grid_place_mapping_spatial.csv').exists():
    subprocess.run([sys.executable, 'build_spatial_grid_place_mapping.py'], check=True)
else:
    print('Using existing data/grid_place_mapping_spatial.csv')

# Build covered-area citywide vitality artifacts from the fresh outputs_colab run.
subprocess.run([sys.executable, 'build_citywide_vitality_artifacts.py'], check=True)

# Build base grid coverage map.
subprocess.run([sys.executable, 'build_base_coverage_map.py'], check=True)


## 7. 결과 파일 확인

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd

with open('api_usage_manifest.json', encoding='utf-8') as f:
    api_manifest = json.load(f)

output_dir = Path(api_manifest['pipeline_output_dir'])
required_outputs = {
    'base_scores': output_dir / 'base_scores.csv',
    'final_scores': output_dir / 'final_scores.csv',
    'metrics_summary': output_dir / 'metrics_summary.json',
    'citywide_final_scores': Path('outputs/citywide_vitality/citywide_final_scores.csv'),
    'citywide_detail_heatmap': Path('outputs/citywide_vitality/citywide_vitality_heatmap_covered_area_detail.png'),
    'coverage_map': Path('outputs/coverage/base_grid_coverage_map.png'),
}

for name, path in required_outputs.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing required output: {name} -> {path}')
    if path.stat().st_size == 0:
        raise ValueError(f'Output file is empty: {name} -> {path}')
    print(f'OK {name}: {path} ({path.stat().st_size:,} bytes)')

base_scores = pd.read_csv(required_outputs['base_scores'])
final_scores = pd.read_csv(required_outputs['final_scores'])
citywide_scores = pd.read_csv(required_outputs['citywide_final_scores'])

if len(base_scores) == 0:
    raise ValueError('base_scores.csv has 0 rows.')
if len(final_scores) == 0:
    raise ValueError('final_scores.csv has 0 rows.')
if len(citywide_scores) == 0:
    raise ValueError('citywide_final_scores.csv has 0 rows.')

if api_manifest['api_mode'] == 'live_api':
    expected_sequence = str(Path('data/correction_infer_live_spatial_sequence_api.csv'))
    actual_sequence = api_manifest['correction_infer_csv']
    if Path(actual_sequence).as_posix() != Path(expected_sequence).as_posix():
        raise ValueError(f'API mode must use generated live sequence. expected={expected_sequence}, actual={actual_sequence}')
    if not api_manifest.get('api_key_set'):
        raise ValueError('API mode is live_api, but api_key_set is false.')

with open(required_outputs['metrics_summary'], encoding='utf-8') as f:
    metrics = json.load(f)

print('\nAPI usage manifest')
print(json.dumps(api_manifest, ensure_ascii=False, indent=2))
print('\nmetrics_summary')
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('\nRow counts')
print({
    'base_scores': len(base_scores),
    'final_scores': len(final_scores),
    'citywide_final_scores': len(citywide_scores),
})

display(final_scores.head())


## 8. 지도/히트맵 이미지 보기

In [ ]:
from IPython.display import Image, display
from pathlib import Path

# final_score_heatmap.png is intentionally omitted: it is a raw latest-timestamp
# diagnostic plot and can collapse to a single point when only one grid exists
# at the latest timestamp. Use the covered-area heatmap for reporting.
image_paths = [
    'outputs/citywide_vitality/citywide_vitality_heatmap_covered_area_detail.png',
    'outputs/coverage/base_grid_coverage_map.png',
]

for p in image_paths:
    if Path(p).exists():
        print(p)
        display(Image(filename=p))


## 9. 실시간 API 입력으로 돌리는 선택 셀

서울 실시간 도시데이터 API 키가 있고, live correction 입력을 새로 만들고 싶을 때만 실행하세요.

In [ ]:
# API note
# The live API path is now handled in section 4 before model execution.
# - USE_LIVE_API=True: fetch Seoul real-time API data, build the LSTM sequence, and run the model with that sequence.
# - USE_LIVE_API=False: use data/correction_infer_live_spatial_sequence.csv for offline reproduction.
# Check api_usage_manifest.json after the run to confirm which path was used.


## 10. 결과 압축 및 다운로드

In [ ]:
from google.colab import files
import shutil
import zipfile
from pathlib import Path

zip_path = Path('/content/seoul_grid_vitality_outputs_colab.zip')
bundle_root = Path('/content/seoul_grid_vitality_outputs_bundle')

if zip_path.exists():
    zip_path.unlink()
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True, exist_ok=True)

def copy_dir(src, dst):
    src_path = Path(src)
    if src_path.exists():
        shutil.copytree(src_path, bundle_root / dst)
        print('Included:', src, '->', bundle_root / dst)
    else:
        raise FileNotFoundError(f'Missing bundle directory: {src}')

copy_dir('outputs_colab', 'outputs_colab')
copy_dir('outputs/citywide_vitality', 'outputs/citywide_vitality')
copy_dir('outputs/coverage', 'outputs/coverage')

manifest_path = Path('api_usage_manifest.json')
if not manifest_path.exists():
    raise FileNotFoundError('api_usage_manifest.json is missing.')
shutil.copy2(manifest_path, bundle_root / 'api_usage_manifest.json')
print('Included:', manifest_path, '->', bundle_root / 'api_usage_manifest.json')

# Exclude raw one-point diagnostic heatmap from the downloadable bundle.
raw_heatmap = bundle_root / 'outputs_colab' / 'final_score_heatmap.png'
if raw_heatmap.exists():
    raw_heatmap.unlink()
    print('Excluded:', raw_heatmap)

shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_root)

required_zip_members = [
    'outputs_colab/final_scores.csv',
    'outputs_colab/metrics_summary.json',
    'outputs/citywide_vitality/citywide_final_scores.csv',
    'outputs/citywide_vitality/citywide_vitality_heatmap_covered_area_detail.png',
    'outputs/coverage/base_grid_coverage_map.png',
    'api_usage_manifest.json',
]
forbidden_zip_members = [
    'outputs_colab/final_score_heatmap.png',
]

with zipfile.ZipFile(zip_path) as zf:
    members = set(zf.namelist())
    print('\nZip contents')
    for member in sorted(members):
        print(member)
    missing = [m for m in required_zip_members if m not in members]
    forbidden = [m for m in forbidden_zip_members if m in members]
    if missing:
        raise FileNotFoundError('Zip is missing required files: ' + ', '.join(missing))
    if forbidden:
        raise ValueError('Zip includes excluded files: ' + ', '.join(forbidden))

files.download(str(zip_path))
